In [1]:
# 02_risk_metrics.ipynb
# Calculate VaR, CVaR, Sharpe ratio, and Max Drawdown

import pandas as pd
import numpy as np
import os

# Load saved data
data_path = os.path.expanduser("~/Desktop/SentriVaR-500/data")
df = pd.read_csv(f"{data_path}/prices.csv", index_col="Date", parse_dates=True)

# Calculate daily returns
returns = df.pct_change().dropna()

print(f"Data loaded: {df.shape}")
print(f"Date range: {df.index[0].date()} ~ {df.index[-1].date()}")

Data loaded: (2141, 5)
Date range: 2018-01-02 ~ 2026-07-10


In [2]:
# Risk metrics as a reusable function
def calculate_risk_metrics(return_series, confidence=0.95):
    """
    Calculate risk metrics for a single asset.
    - return_series: daily returns Series
    - confidence: VaR confidence level (default 95%)
    """
    # VaR: max expected daily loss at the given confidence level
    VaR = return_series.quantile(1 - confidence)

    # CVaR: average loss beyond the VaR threshold
    CVaR = return_series[return_series <= VaR].mean()

    # Sharpe ratio: risk-adjusted return, annualized
    sharpe = return_series.mean() / return_series.std() * np.sqrt(252)

    # MDD: max drawdown from peak
    cumulative = (1 + return_series).cumprod()
    rolling_max = cumulative.cummax()
    drawdown = (cumulative - rolling_max) / rolling_max
    mdd = drawdown.min()

    return {
        "VaR (95%)": round(VaR, 4),
        "CVaR (95%)": round(CVaR, 4),
        "Sharpe Ratio": round(sharpe, 4),
        "MDD": round(mdd, 4)
    }

# Apply to all tickers
results = {}
for ticker in returns.columns:
    results[ticker] = calculate_risk_metrics(returns[ticker])

risk_df = pd.DataFrame(results).T
print(risk_df)

       VaR (95%)  CVaR (95%)  Sharpe Ratio     MDD
AAPL     -0.0297     -0.0438        0.9465 -0.3852
GOOGL    -0.0296     -0.0443        0.8790 -0.4432
JPM      -0.0264     -0.0410        0.7044 -0.4363
MSFT     -0.0283     -0.0407        0.7952 -0.3715
SOXX     -0.0367     -0.0520        0.9548 -0.4575


In [3]:
# Save risk metrics
risk_df.to_csv(f"{data_path}/risk_metrics.csv")
print("Saved: risk_metrics.csv")

# Quick summary
print("\n--- Per-ticker breakdown ---")
print(f"Best risk-adjusted return: {risk_df['Sharpe Ratio'].idxmax()} (Sharpe {risk_df['Sharpe Ratio'].max()})")
print(f"Lowest VaR: {risk_df['VaR (95%)'].idxmax()} (VaR {risk_df['VaR (95%)'].max()})")
print(f"Smallest drawdown: {risk_df['MDD'].idxmax()} (MDD {risk_df['MDD'].max()})")

Saved: risk_metrics.csv

--- Per-ticker breakdown ---
Best risk-adjusted return: SOXX (Sharpe 0.9548)
Lowest VaR: JPM (VaR -0.0264)
Smallest drawdown: MSFT (MDD -0.3715)
